# Notebook 09 — LSTM

Este notebook implementa una red neuronal LSTM (Long Short-Term Memory)
para la predicción de la huella de carbono operacional del sistema
eléctrico español con resolución de 15 minutos.

El LSTM extiende la RNN simple añadiendo tres puertas de memoria
explícitas — forget, input y output — que permiten al modelo
aprender qué información retener y qué olvidar a lo largo de
secuencias largas, resolviendo el problema del vanishing gradient
de la RNN simple.

**Evaluación:** walk-forward con 12 fits distribuidos por 2024,
misma configuración que el resto de modelos.

**Horizontes:** 48h (192 pasos) y 72h (288 pasos).

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
%matplotlib inline

BASE_DIR    = Path("/home/ubuntu/TFM")
DATA_DIR    = BASE_DIR / "notebooks/data_processed"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Librerías cargadas OK")
print("Device:", DEVICE)

Librerías cargadas OK
Device: cpu


## 1. Carga de datos

In [2]:
y_train = pd.read_parquet(DATA_DIR / "train_2022_2023.parquet")["y"].astype(float)
y_val   = pd.read_parquet(DATA_DIR / "val_2024.parquet")["y"].astype(float)
y_test  = pd.read_parquet(DATA_DIR / "test_2025.parquet")["y"].astype(float)

y_train.index = pd.to_datetime(y_train.index)
y_val.index   = pd.to_datetime(y_val.index)
y_test.index  = pd.to_datetime(y_test.index)

print("Train:", y_train.shape, "|", y_train.index.min(), "->", y_train.index.max())
print("Val:  ", y_val.shape,   "|", y_val.index.min(),   "->", y_val.index.max())
print("Test: ", y_test.shape,  "|", y_test.index.min(),  "->", y_test.index.max())

Train: (70080,) | 2022-01-01 00:00:00+00:00 -> 2023-12-31 23:45:00+00:00
Val:   (35136,) | 2024-01-01 00:00:00+00:00 -> 2024-12-31 23:45:00+00:00
Test:  (35040,) | 2025-01-01 00:00:00+00:00 -> 2025-12-31 23:45:00+00:00


In [3]:
FREQ_MIN        = 15
STEPS_PER_HOUR  = 60 // FREQ_MIN
SEASONAL_PERIOD = 24 * STEPS_PER_HOUR  # 96

HORIZONS = {
    "48h": 48 * STEPS_PER_HOUR,  # 192 pasos
    "72h": 72 * STEPS_PER_HOUR,  # 288 pasos
}

LOOKBACK    = 6 * SEASONAL_PERIOD  # 6 días = 576 pasos
HIDDEN_SIZE = 64
NUM_LAYERS  = 1
BATCH_SIZE  = 128
EPOCHS      = 50
LR          = 1e-3

torch.manual_seed(42)
np.random.seed(42)

print("Periodo estacional:", SEASONAL_PERIOD)
print("Lookback:", LOOKBACK, "pasos =", LOOKBACK // SEASONAL_PERIOD, "días")
print("Device:", DEVICE)

Periodo estacional: 96
Lookback: 576 pasos = 6 días
Device: cpu


## 3. Normalización

In [ ]:
scaler = MinMaxScaler()
scaler.fit(y_train.values.reshape(-1, 1))

y_train_scaled = scaler.transform(y_train.values.reshape(-1, 1)).flatten()
y_val_scaled   = scaler.transform(y_val.values.reshape(-1, 1)).flatten()
y_test_scaled  = scaler.transform(y_test.values.reshape(-1, 1)).flatten()

print("Train — min:", y_train_scaled.min().round(3), "max:", y_train_scaled.max().round(3))
print("Val   — min:", y_val_scaled.min().round(3),   "max:", y_val_scaled.max().round(3))
print("Test  — min:", y_test_scaled.min().round(3),  "max:", y_test_scaled.max().round(3))

## 4. Dataset y DataLoaders

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, series, lookback, horizon):
        self.X, self.y = [], []
        for i in range(lookback, len(series) - horizon + 1):
            self.X.append(series[i - lookback:i])
            self.y.append(series[i:i + horizon])
        self.X = torch.tensor(np.array(self.X), dtype=torch.float32).unsqueeze(-1)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

datasets = {}
loaders  = {}

for name, h in HORIZONS.items():
    ds_train = SlidingWindowDataset(y_train_scaled, LOOKBACK, h)
    ds_val   = SlidingWindowDataset(
        np.concatenate([y_train_scaled[-LOOKBACK:], y_val_scaled]),
        LOOKBACK, h
    )
    datasets[name] = {"train": ds_train, "val": ds_val}
    loaders[name]  = {
        "train": DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True),
        "val":   DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False)
    }
    print(f"{name} — train: {len(ds_train)} | val: {len(ds_val)}")

## 5. Arquitectura LSTM

El LSTM extiende la RNN simple incorporando tres puertas de memoria:
- **Forget gate**: decide qué información del estado anterior olvidar
- **Input gate**: decide qué información nueva almacenar
- **Output gate**: decide qué parte del estado de memoria usar como salida

Estas puertas permiten al modelo aprender dependencias de largo
alcance sin el problema del vanishing gradient de la RNN simple,
lo que lo hace especialmente adecuado para series temporales
con patrones de largo alcance como la huella de carbono.

La arquitectura consta de:
- Una capa LSTM con HIDDEN_SIZE unidades ocultas
- Una capa fully connected que mapea el último estado oculto
  a los H pasos de predicción

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, horizon=192):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

# Test rápido
for name, h in HORIZONS.items():
    model  = LSTMModel(hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, horizon=h)
    x_test = torch.randn(4, LOOKBACK, 1)
    y_test = model(x_test)
    print(f"{name} — input: {x_test.shape} → output: {y_test.shape}")

## 6. Training loop

El training loop entrena el modelo minimizando el MSE entre
las predicciones y los valores reales usando el optimizador Adam.
En cada época se calcula el error sobre train y validación para
monitorizar la convergencia y detectar sobreajuste.

In [ ]:
def train_model(model, loader_train, loader_val, epochs, lr, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    model.to(device)

    history = {"train_loss": [], "val_loss": []}

    for epoch in tqdm(range(1, epochs + 1), desc="Entrenando"):
        model.train()
        train_loss = 0
        for X_batch, y_batch in loader_train:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(X_batch)
        train_loss /= len(loader_train.dataset)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in loader_val:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                pred = model(X_batch)
                loss = criterion(pred, y_batch)
                val_loss += loss.item() * len(X_batch)
        val_loss /= len(loader_val.dataset)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if epoch % 5 == 0:
            tqdm.write(f"Epoch {epoch}/{epochs} — train: {train_loss:.4f} | val: {val_loss:.4f}")

    return history

print("Training loop definido OK")

## 7. Entrenamiento del modelo

Se entrena un modelo LSTM independiente para cada horizonte.

**Hiperparámetros seleccionados:**
- LOOKBACK=576 -> 6 días de contexto, igual que AutoReg para comparación justa
- HIDDEN_SIZE=64 -> tamaño del estado oculto
- NUM_LAYERS=1 -> una sola capa LSTM
- EPOCHS=50 -> número de épocas de entrenamiento
- LR=1e-3 -> learning rate del optimizador Adam
- BATCH_SIZE=128 -> ejemplos por actualización de pesos

Estos hiperparámetros son un punto de partida — se optimizarán
en la siguiente sección si los resultados no son satisfactorios.

In [ ]:
models_lstm = {}
histories   = {}

for name, h in HORIZONS.items():
    print(f"\nEntrenando LSTM para horizonte {name}...")
    torch.manual_seed(42)

    model = LSTMModel(
        input_size=1,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        horizon=h
    )

    history = train_model(
        model,
        loaders[name]["train"],
        loaders[name]["val"],
        epochs=EPOCHS,
        lr=LR,
        device=DEVICE
    )

    models_lstm[name] = model
    histories[name]   = history
    print(f"Entrenamiento {name} completado")